# 08 — Đánh giá trên test độc lập
Đây là lần **đầu tiên** tập test được dùng, với mô hình đã chốt ở bước 07 (không chỉnh lại sau khi xem test). So sánh với các phương pháp chuẩn: PBE (không hiệu chỉnh), LDA (không tự hợp), và hiệu chỉnh hằng số bằng trung bình train.

**Đầu ra:** `data/processed/test_results.csv`, `test_metrics.json`, `test_parity.png`.

In [ ]:
# Cài thư viện (chỉ chạy trên Colab; runtime Colab bị reset sau mỗi phiên nên mỗi notebook tự cài thứ nó cần)
import sys
if "google.colab" in sys.modules:
    %pip install -q h5py optax matplotlib

In [ ]:
import os, sys, json, time
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ml_xc_colab"
else:
    ROOT = os.environ.get("ML_XC_ROOT", os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)          # để import được thư mục utils/
DATA, RAW, PROC = f"{ROOT}/data", f"{ROOT}/data/raw", f"{ROOT}/data/processed"
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from utils import data_utils as dtu, density_utils as du, model_utils as mu

H5 = f"{DATA}/ml_xc_dataset.h5"
params, cfg = mu.load_model(f"{DATA}/models/best_model.npz")
print("Mô hình:", {k: cfg[k] for k in ["hidden", "lr", "val_rmse_meV"]})

ids = dtu.list_materials(H5, "test")
test_ds = mu.make_dataset([dtu.load_sample(H5, m) for m in ids], du.ml_features)
tr = dtu.read_targets(H5, "train")
shift = float(tr.delta_ref_pbe_eV_per_atom.mean())          # chỉ lấy thống kê từ train
print(f"Test: {len(test_ds)} mẫu | hiệu chỉnh hằng số từ train = {shift*1e3:.1f} meV/nguyên tử")

In [ ]:
target = np.array([d["target_eV"] for d in test_ds])          # Δ_ref = E_ref - E_PBE (eV/nguyên tử)
pred = {
    "ML-XC":          mu.predict(params, test_ds),
    "PBE (Δ=0)":      np.zeros(len(test_ds)),
    "LDA (không SCF)": np.array([d["lda_eV"] for d in test_ds]),
    "PBE + hằng số train": np.full(len(test_ds), shift),
}
table = pd.DataFrame({name: mu.metrics(p, target) for name, p in pred.items()}).T[["mae_meV", "rmse_meV", "max_meV", "n"]]
print("Sai số so với nhãn tham chiếu trên TEST (meV/nguyên tử):")
table.round(2)

In [ ]:
t = dtu.read_targets(H5, "test").set_index("material_id").loc[ids]
res = t.assign(target_eV=target, **{f"pred_{k}": v for k, v in pred.items()})
res["err_ML_meV"] = (pred["ML-XC"] - target) * 1e3
res["is_metal"] = res.band_gap_MP_eV < 0.5
res.to_csv(f"{PROC}/test_results.csv")

by = {}
for name, p in pred.items():
    for flag, g in res.groupby("is_metal"):
        by.setdefault(("kim loại" if flag else "không kim loại"), {})[name] = mu.metrics(p[res.is_metal.values == flag],
                                                                                         target[res.is_metal.values == flag])["rmse_meV"]
print("RMSE theo nhóm (meV/nguyên tử):"); print(pd.DataFrame(by).round(2))
json.dump({"overall": table.to_dict("index"), "by_group_rmse_meV": by, "n_test": len(ids),
           "model": cfg}, open(f"{PROC}/test_metrics.json", "w"), indent=2, default=float)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, name in zip(ax, ["PBE (Δ=0)", "ML-XC"]):
    a.scatter(target * 1e3, pred[name] * 1e3, s=14, alpha=.7)
    lim = [min(target.min(), pred[name].min()) * 1e3, max(target.max(), pred[name].max()) * 1e3]
    a.plot(lim, lim, "k--", lw=1); a.set_title(name)
    a.set_xlabel("Δ tham chiếu (meV/nguyên tử)"); a.set_ylabel("Δ dự đoán (meV/nguyên tử)")
plt.tight_layout(); plt.savefig(f"{PROC}/test_parity.png", dpi=120); plt.show()

### Diễn giải
- Nếu ML-XC không thắng baseline **PBE + hằng số train**, mô hình chưa học được gì ngoài độ lệch trung bình.
- Số mẫu test nhỏ nên các chỉ số có độ bất định lớn; xem `max_meV` và nhóm kim loại/không kim loại, đừng chỉ nhìn một con số tổng.
- Nhãn có sai số riêng từ đánh giá không tự hợp (xem `labels_method.json`); sai số mô hình nhỏ hơn mức này không còn ý nghĩa.